# 1. KQL Hunting Patterns

KQL (Kusto Query Language) is the language of Sentinel, Defender XDR, Azure Monitor, and Azure Data Explorer. SC-200 requires you to write and interpret KQL queries.

## KQL basics → SIEM query mapping

Our mini-SIEM API maps to KQL concepts:

| KQL | Our SIEM API equivalent |
|-----|------------------------|
| `SigninLogs` | `table_name: 'SigninLogs'` |
| `\| where ResultType == 'Failure'` | `filter: {ResultType: 'Failure'}` |
| `\| summarize count() by UserPrincipalName` | `aggregate_by: 'UserPrincipalName'` |
| `\| where TimeGenerated > ago(1h)` | `time_range_minutes: 60` |
| `\| take 10` | `limit: 10` |

We'll write queries in both formats so you learn KQL while using the SIEM.

In [ ]:
import httpx, json
from collections import Counter, defaultdict

SIEM = 'http://localhost:8000'

def query(table, filter=None, aggregate_by=None, time_range=None, limit=100):
    r = httpx.post(f'{SIEM}/query', json={
        'table_name': table,
        'filter': filter,
        'aggregate_by': aggregate_by,
        'time_range_minutes': time_range,
        'limit': limit,
    })
    return r.json()['results']

# ----- Hunt 1: Who has the most failed sign-ins? -----
print('=== Hunt 1: Accounts with most failed sign-ins ===')
print('KQL: SigninLogs | where ResultType == "Failure" | summarize count() by UserPrincipalName | sort by count_ desc\n')

results = query('SigninLogs', filter={'ResultType': 'Failure'}, aggregate_by='UserPrincipalName')
for r in results:
    bar = '█' * min(r['count'], 40)
    print(f'  {r["group_key"]:<30} {r["count"]:>3} {bar}')

In [ ]:
# ----- Hunt 2: Where are sign-ins coming from? -----
print('=== Hunt 2: Sign-in locations ===')
print('KQL: SigninLogs | summarize count() by Location | sort by count_ desc\n')

results = query('SigninLogs', aggregate_by='Location')
suspicious_locs = {'Moscow', 'Beijing', 'Anonymous Proxy'}
for r in results:
    flag = ' ⚠️' if r['group_key'] in suspicious_locs else ''
    print(f'  {r["group_key"]:<20} {r["count"]:>3}{flag}')

In [ ]:
# ----- Hunt 3: Rare processes on endpoints -----
print('=== Hunt 3: Rare processes (potential LOLBins or attack tools) ===')
print('KQL: DeviceEvents | summarize count() by FileName | where count_ <= 3 | sort by count_ asc\n')

results = query('DeviceEvents', aggregate_by='FileName')
known_attack_tools = {'mimikatz.exe', 'psexec.exe', 'certutil.exe', 'curl', 'powershell.exe'}

# Show rare processes (low count = unusual = interesting)
rare = [r for r in results if r['count'] <= 5]
for r in sorted(rare, key=lambda x: x['count']):
    is_tool = '🔴 ATTACK TOOL' if r['group_key'] in known_attack_tools else ''
    print(f'  {r["group_key"]:<20} seen {r["count"]} time(s)  {is_tool}')

In [ ]:
# ----- Hunt 4: Outbound connections to unusual destinations -----
print('=== Hunt 4: Outbound traffic analysis ===')
print('KQL: AzureFirewall | summarize ConnectionCount=count() by DestinationIP | where DestinationIP !startswith "10."\n')

results = query('AzureFirewall', aggregate_by='DestinationIP')
known_bad = {'185.220.101.42', '45.33.32.156', '198.51.100.99'}

for r in results:
    ip = r['group_key']
    if ip and not ip.startswith('10.'):
        threat = '🔴 KNOWN BAD' if ip in known_bad else '🟡 External'
        print(f'  {ip:<20} {r["count"]:>3} connections  {threat}')

In [ ]:
# ----- Hunt 5: Successful sign-ins after failures (credential stuffing success) -----
print('=== Hunt 5: Users with failures followed by success (compromised accounts) ===')
print('KQL equivalent:')
print('  let failures = SigninLogs | where ResultType == "Failure" | summarize FailCount=count() by UserPrincipalName;')
print('  let successes = SigninLogs | where ResultType == "Success" | summarize SuccessCount=count() by UserPrincipalName;')
print('  failures | join successes on UserPrincipalName | where FailCount > 5\n')

# Get all sign-ins
all_signins = query('SigninLogs', limit=500)
user_stats = defaultdict(lambda: {'failures': 0, 'successes': 0, 'ips': set()})

for s in all_signins:
    user = s['UserPrincipalName']
    if s['ResultType'] == 'Failure':
        user_stats[user]['failures'] += 1
    else:
        user_stats[user]['successes'] += 1
    user_stats[user]['ips'].add(s['IPAddress'])

for user, stats in user_stats.items():
    if stats['failures'] > 5 and stats['successes'] > 0:
        print(f'  🔴 {user}: {stats["failures"]} failures + {stats["successes"]} successes')
        print(f'     IPs: {stats["ips"]}')
        print(f'     → Likely COMPROMISED — investigate immediately!\n')

## Real KQL operators for the SC-200 exam

| Operator | What it does | Example |
|----------|-------------|----------|
| `where` | Filter rows | `\| where ResultType == "Failure"` |
| `summarize` | Aggregate | `\| summarize count() by UserPrincipalName` |
| `project` | Select columns | `\| project TimeGenerated, User, IP` |
| `extend` | Add computed column | `\| extend Hour = bin(TimeGenerated, 1h)` |
| `join` | Combine tables | `T1 \| join T2 on UserId` |
| `union` | Stack tables vertically | `SigninLogs \| union DeviceLogonEvents` |
| `sort by` | Order results | `\| sort by count_ desc` |
| `top` | First N rows | `\| top 10 by count_` |
| `ago()` | Time filter | `\| where TimeGenerated > ago(1h)` |
| `bin()` | Time bucketing | `bin(TimeGenerated, 5m)` |
| `render` | Visualization | `\| render timechart` |
| `let` | Variable | `let threshold = 5;` |
| `externaldata` | Load external data | CSV, JSON files |
| `mv-expand` | Expand arrays | Unpack multi-value fields |

### Key KQL tables for SC-200

| Table | Product | Contains |
|-------|---------|----------|
| `SigninLogs` | Entra ID | User sign-in events |
| `AuditLogs` | Entra ID | Directory changes |
| `DeviceEvents` | Defender for Endpoint | Process, file, registry events |
| `DeviceNetworkEvents` | Defender for Endpoint | Network connections |
| `EmailEvents` | Defender for Office 365 | Email delivery |
| `UrlClickEvents` | Defender for Office 365 | Safe Links clicks |
| `CloudAppEvents` | Defender for Cloud Apps | SaaS activity |
| `IdentityLogonEvents` | Defender for Identity | On-prem AD logons |
| `SecurityAlert` | Sentinel | All alerts |
| `SecurityIncident` | Sentinel | All incidents |

**Next**: [Notebook 2 — Advanced Threat Hunting](02_advanced_hunting.ipynb)